<a href="https://colab.research.google.com/github/buse-toklu/CENG467_Midterm/blob/main/5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets  # Installing HuggingFace datasets library

import math
import random
import time
import warnings
from collections import Counter, defaultdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")

# Reproducibility settings
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device being used: {DEVICE}")

In [ ]:
def load_wikitext2() -> Tuple[List[str], List[str], List[str]]:
    from datasets import load_dataset
    print("Loading WikiText-2 dataset...")
    ds = load_dataset("wikitext", "wikitext-2-raw-v1", trust_remote_code=True)

    def _tokenize(split):
        tokens = []
        for row in ds[split]["text"]:
            row = row.strip()
            if row:
                tokens.extend(row.lower().split())
        return tokens

    return _tokenize("train"), _tokenize("validation"), _tokenize("test")

train_tok, valid_tok, test_tok = load_wikitext2()
print(f"Tokens - Train: {len(train_tok)}, Valid: {len(valid_tok)}, Test: {len(test_tok)}")

In [ ]:
class Vocabulary:
    PAD, UNK, BOS, EOS = "<pad>", "<unk>", "<bos>", "<eos>"
    def __init__(self, tokens: List[str], min_freq: int = 3):
        special = [self.PAD, self.UNK, self.BOS, self.EOS]
        freq = Counter(tokens)
        vocab_tokens = special + sorted(t for t, c in freq.items() if c >= min_freq)
        self.token2idx = {t: i for i, t in enumerate(vocab_tokens)}
        self.idx2token = {i: t for t, i in self.token2idx.items()}
        self.pad_idx = self.token2idx[self.PAD]
        self.unk_idx = self.token2idx[self.UNK]
        self.bos_idx = self.token2idx[self.BOS]
        self.eos_idx = self.token2idx[self.EOS]

    def __len__(self) -> int: return len(self.token2idx)
    def encode(self, tokens: List[str]) -> List[int]:
        return [self.token2idx.get(t, self.unk_idx) for t in tokens]
    def decode(self, indices: List[int]) -> List[str]:
        return [self.idx2token.get(i, self.UNK) for i in indices]

vocab = Vocabulary(train_tok)
print(f"Vocabulary Size: {len(vocab)}")

In [ ]:
class NGramModel:
    def __init__(self, n: int = 2, k: float = 1.0):
        self.n = n
        self.k = k
        self.counts = defaultdict(Counter)
        self.vocab_size = 0

    def train(self, tokens: List[str], vocab: Vocabulary):
        self.vocab_size = len(vocab)
        ids = vocab.encode(tokens)
        for i in range(len(ids) - self.n + 1):
            context = tuple(ids[i: i + self.n - 1])
            next_id = ids[i + self.n - 1]
            self.counts[context][next_id] += 1

    def _log_prob(self, context: tuple, next_id: int) -> float:
        ctx_count = sum(self.counts[context].values())
        word_count = self.counts[context][next_id]
        prob = (word_count + self.k) / (ctx_count + self.k * self.vocab_size)
        return math.log(prob)

    def perplexity(self, tokens: List[str], vocab: Vocabulary) -> float:
        ids = vocab.encode(tokens)
        total_log_prob, count = 0.0, 0
        for i in range(len(ids) - self.n + 1):
            context = tuple(ids[i: i + self.n - 1])
            next_id = ids[i + self.n - 1]
            total_log_prob += self._log_prob(context, next_id)
            count += 1
        return math.exp(-total_log_prob / count)

    def generate(self, vocab: Vocabulary, seed: List[str], max_len: int = 20) -> str:
        ids = vocab.encode(seed)
        context = tuple(ids[-(self.n-1):])
        generated = list(ids)
        for _ in range(max_len):
            ctx_dict = self.counts[context]
            ctx_total = sum(ctx_dict.values()) + self.k * self.vocab_size
            probs = [(ctx_dict[w] + self.k) / ctx_total for w in range(self.vocab_size)]
            next_id = np.random.choice(len(probs), p=probs)
            generated.append(next_id)
            context = tuple(generated[-(self.n-1):])
        return " ".join(vocab.decode(generated))

trigram = NGramModel(n=3)
trigram.train(train_tok, vocab)
print(f"Trigram Test Perplexity: {trigram.perplexity(test_tok, vocab):.2f}")

In [ ]:
class LSTMDataset(Dataset):
    def __init__(self, token_ids: List[int], seq_len: int = 35):
        self.data = torch.tensor(token_ids, dtype=torch.long)
        self.seq_len = seq_len
    def __len__(self) -> int: return max(0, len(self.data) - self.seq_len)
    def __getitem__(self, idx: int):
        return self.data[idx: idx+self.seq_len], self.data[idx+1: idx+self.seq_len+1]

class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        out, hidden = self.lstm(self.embedding(x), hidden)
        return self.fc(out), hidden

In [ ]:
def train_lstm(model, train_ids, vocab, epochs=3):
    train_ds = LSTMDataset(train_ids)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad_idx)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits, _ = model(x)
            loss = criterion(logits.view(-1, len(vocab)), y.view(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch} | Train PPL: {math.exp(total_loss/len(train_loader)):.2f}")

def generate_lstm(model, vocab, seed, max_len=20):
    model.eval()
    ids = vocab.encode(seed)
    input_seq = torch.tensor([ids], device=DEVICE)
    generated, hidden = list(ids), None
    logits, hidden = model(input_seq, hidden)
    next_id = torch.multinomial(torch.softmax(logits[0, -1], dim=-1), 1).item()
    for _ in range(max_len):
        generated.append(next_id)
        input_t = torch.tensor([[next_id]], device=DEVICE)
        logits, hidden = model(input_t, hidden)
        next_id = torch.multinomial(torch.softmax(logits[0, -1], dim=-1), 1).item()
    return " ".join(vocab.decode(generated))

lstm_model = LSTMLanguageModel(len(vocab)).to(DEVICE)
train_lstm(lstm_model, vocab.encode(train_tok), vocab, epochs=3)

In [ ]:
seeds = [["the", "university"], ["in", "the"]]

print("--- Comparison of Generated Text ---")
for s in seeds:
    print(f"\nSeed: {' '.join(s)}")
    print(f"Trigram: {trigram.generate(vocab, s)}")
    print(f"LSTM:    {generate_lstm(lstm_model, vocab, s)}")

print("\n--- Brief Discussion ---")
print("1. The N-gram model only considers local context (the last 2 words), leading to limited grammatical structure.")
print("2. The LSTM model produces more fluent sentences by maintaining a hidden state that captures longer dependencies.")
print("3. Perplexity values show that the LSTM achieves a significantly better fit for the WikiText-2 dataset patterns.")